# 手寫數字分類

課程提交版本的公開整理副本。所有歷史輸出、附件及非必要 metadata 已移除。
先閱讀上層 README 的資料需求、已知問題與重現限制；不要直接 Run All。


# Homework 1：簡易手寫數字辨識


## Import


In [ ]:
# For data loading and scientific computing
import pandas as pd
import numpy as np
import random
import os
import csv

# For data visualization
import matplotlib.pyplot as plt
import plotly.express as px

# Deep learning framework
import torch
import torch.nn as nn
import torch.nn.functional as F
# data processing and evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

import warnings
warnings.filterwarnings('ignore')

## Configuration


In [ ]:
# TODO_1: 調整適當的參數
class config:
    root = "kaggle/input/mia-hw-1"
    batch_size = 16      # 通常是2的倍數
    lr = 0.001          # 通常是10的倍數，1e-8 = 10^-8 0.001
    epochs = 25          # 看你心情要設多少
    weight_decay = 0.01  #對loss funtion的懲罰項，模型越複雜，loss會越大，有防止overfitting的用意。
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seed = 7            # 看你心情要設多少

## Utils


## EDA


In [ ]:
def seed_everything(seed):
    # Set Python random seed
    random.seed(seed)

    # Set NumPy random seed
    np.random.seed(seed)

    # Set PyTorch random seed for CPU and GPU
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Set PyTorch deterministic operations for cudnn backend
    torch.backends.cudnn.deterministic = True  #True:確保相同輸入有相同輸出
    torch.backends.cudnn.benchmark = False     #True:自動尋找最佳算法，提升運算效率，但這裡我們設成 False 。

In [ ]:
# Read in the dataframe
train_data = pd.read_csv(os.path.join(config.root, 'zip.train.csv'), header=None)
test_data = pd.read_csv(os.path.join(config.root, 'zip.test.csv'), header=None)
print(train_data)

In [ ]:
train_data.iloc[:, 2:].describe() #總結資料集分佈的集中趨勢、離散度和形狀（不包括 NaN值）

In [ ]:
print(f"Number of training samples: {train_data.shape[0]}")
print(f"Number of testing samples: {test_data.shape[0]}")

In [ ]:
train_data.iloc[:, 1].value_counts().sort_index() #計算train_data標籤數量

## Show the number of images in each digit for the training dataset (2 pts)


In [ ]:
train_label_distribution = train_data.iloc[:, 1].value_counts().sort_index()
train_label_distribution = pd.DataFrame(
    {
        "labels": train_label_distribution.index.astype(str), #將int64改成str
        "number_of_samples": train_label_distribution.values
    }
)

# TODO_2: 試著畫出每一個類別的樣本數量，填寫 yaxis=

#以下兩個是視覺化參數設定
fig = px.bar(
    train_label_distribution, x="labels", y="number_of_samples",
    labels=dict(y="number of samples"),
)
# update_layout 用來更新圖表的佈局配置，調整圖表的外觀和樣式
fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='lightgray', zeroline=False),  # Show x-axis grid，zeroline=False為不顯示x=0那條線
    yaxis=dict(showgrid=True, gridcolor='lightgray', zeroline=False),                                                            # Show y-axis grid，zeroline=False為不顯示y=0那條線
    title=dict(text="Number of Samples Per Labels", x=0.5, y=0.975, font=dict(size=20))
)
fig.show()

## Pytorch Dataset


In [ ]:
class HandwrittenDigits(torch.utils.data.Dataset):
    def __init__(self, df, mode = None, transforms=None):
        self.df = df
        self.transforms = transforms #之後的作業會用到
        self.mode = mode

    def __len__(self):
        return len(self.df)

    def __getitem__(self, x):
        if self.mode == "train":
            ID = self.df.iloc[x, 0]                                 #抓取第 x 筆資料，第 0 欄為 image_id。
            label = self.df.iloc[x, 1]                              #抓取第 x 筆資料，第 1 欄為 image_label。
            image = self.df.iloc[x, 2:].to_numpy(dtype=np.float32)  #抓取第 x 筆資料，第 2 欄為 image 的 pixel。
            label, image = torch.tensor(label), torch.tensor(image)

            if self.transforms: #這個只會用在 train_data 身上，所以 valid_data 不用。
                transformed = self.transforms(image=image)
                image = transformed['image']

            return ID, label, image

        # TODO_3: 寫出當 mode = valid、test 的 getitem()
        elif self.mode == "valid":
            ID = self.df.iloc[x, 0]
            label = self.df.iloc[x, 1]
            label, image = self.df.iloc[x, 2:].to_numpy(dtype=np.float32)

            return ID, label, image

        elif self.mode == "test":
            ID = self.df.iloc[x, 0]
            image = self.df.iloc[x, 1:].to_numpy(dtype=np.float32)
            image = torch.tensor(image, dtype=torch.float32)  # 僅轉換 image，因為 test 沒有 label

            return ID, image  # 因為 test 沒 label

In [ ]:
ds = HandwrittenDigits(train_data, mode = "train")
ds[0]

## Pytorch DataLoader


In [ ]:
train_data = pd.read_csv(os.path.join(config.root, 'zip.train.csv'), header=None)
train_dataset = HandwrittenDigits(train_data, mode = "train")
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True) #shuffle=True代表使用 RandomSampler。

## For each digit, plot 10 randomly selected images from your training dataset. (4 pts)


In [ ]:
# TODO_4: 將所看到的空格都填滿，總共要填三個部分: range()、reshape(, )、axes[ , ]

# 建立字典用來儲存 0~9 的圖像
digit_images = {i: [] for i in range(10)} # range()裡面要填數字

# 從train_dataset抓取 label 跟 image，並將 image 重塑成 16x16，放進字典中。
for ID, label, image in train_dataset:
    digit_images[label.item()].append(image.numpy().reshape(16, 16)) # reshape( , )裡面要填兩個數字

# 繪製每個數字的隨機10張圖片
fig, axes = plt.subplots(10, 10, figsize=(10, 10))
fig.suptitle('10 Randomly Selected Images for Each Digit', fontsize=16)

for digit in range(10):
    # 隨機選擇10張圖片
    selected_images = random.sample(digit_images[digit], 10)
    for i, img in enumerate(selected_images):
        ax = axes[digit,i]                                   #axes 是一個 10x10 的 matplotlib 子圖， 其[ , ]要填兩個變數來鎖定子圖位子
        ax.imshow(img, cmap='gray')
        ax.axis('off')

plt.tight_layout() #自動調整圖片間距，避免圖像太過緊密。
plt.show()

# Pytorch Module


In [ ]:
# TODO_5: 建立自己喜歡的 model 架構
# 這裡不需要有 softmax，因為之後的 nn.CrossEntropyLoss()隱含了一個 softmax ，如果在這裡加 softmax，你會套用到兩次。
class SimpleNN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_dim, 1024)
        self.bn1 = nn.BatchNorm1d(1024)
        self.relu1 = nn.ReLU(inplace=True)

        self.fc2 = nn.Linear(1024, 512)
        self.bn2 = nn.BatchNorm1d(512)
        self.relu2 = nn.ReLU(inplace=True)

        self.dropout1 = nn.Dropout(0.1)

        self.fc3 = nn.Linear(512, 512)
        self.bn3 = nn.BatchNorm1d(512)
        self.relu3 = nn.ReLU(inplace=True)

        self.dropout2 = nn.Dropout(0.1)

        self.fc4 = nn.Linear(512, 512)
        self.bn4 = nn.BatchNorm1d(512)
        self.relu4 = nn.ReLU(inplace=True)

        self.dropout3 = nn.Dropout(0.1)

        self.fc5 = nn.Linear(512, 256)
        self.bn5 = nn.BatchNorm1d(256)
        self.relu5 = nn.ReLU(inplace=True)

        self.final_fc = nn.Linear(256, output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu1(x)

        x = self.fc2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.dropout1(x)

        x = self.fc3(x)
        x = self.bn3(x)
        x = self.relu3(x)
        x = self.dropout2(x)

        x = self.fc4(x)
        x = self.bn4(x)
        x = self.relu4(x)
        x = self.dropout3(x)

        x = self.fc5(x)
        x = self.bn5(x)
        x = self.relu5(x)
   
        x = self.final_fc(x)
        return x

In [ ]:
input_dim =  256 # image 的 pixel 有 256
output_dim = 10  # 我們想要分類 0~9 其中一個類別

model = SimpleNN(input_dim, output_dim)
print(model)

In [ ]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"總可訓練參數數量: {total_params}")

## Evaluator


In [ ]:
# TODO_6: 嘗試去計算 misclassification_rate
def evaluator(preds, gts):
    preds = preds.cpu().numpy() if isinstance(preds, torch.Tensor) else preds # 將 Tensor 轉成 numpy ，如果是 Tensor 的話
    gts = gts.cpu().numpy() if isinstance(gts, torch.Tensor) else gts

    acc = accuracy_score(preds, gts)             # 計算正確率
    f1 = f1_score(preds, gts, average="macro")   # F1 score，介在 0~1 ，對 model 分類性能的綜合評價，越高越好
    misclassification_rate = 1 - acc                    # 計算分錯率

    return acc, f1, misclassification_rate

# Training


In [ ]:
def train_one_epoch(model, train_loader, optimizer, scheduler, criterion, device):
    model.train()                                            # PyTorch 中一個用來告知 model 進行訓練的狀態設置，會啟用一些用在訓練的層（如 dropout 和 batch normalization）
    train_loss = .0                                          # 累積一個 epoch 的總損失
    predictions, ground_truths = [], []                      # 儲存所有的預測結果和對應的真實標籤。
    for ID, labels, images in train_loader:
        labels = labels.to(device=device, dtype=torch.long)  # nn.CrossEntropyLoss 要求 label 必須是長整數型態，一般的 int 不行。
        images = images.to(device=device, dtype=torch.float) # 將 image 移動到指定的設備（例如 GPU）上，並確保其資料型態為 float。

        optimizer.zero_grad()                                # 清空 optimizer 的梯度，避免梯度累加
        logits = model(images)                               # 將 image 輸入 model ，計算輸出的 logits。
        loss = criterion(logits, labels)                     # 計算 model 輸出與真實標籤之間的損失（loss）
        loss.backward()                                      # 計算 loss 對 model 參數的梯度

        optimizer.step()                                     # 更新 model 參數

        train_loss += loss.item()                            # 將當前 batch 的損失累加到總損失中。
        preds = torch.argmax(logits, dim=1)                  # 獲取機率最大的類別，當作預測

        predictions.append(preds)                            # 新增預測結果
        ground_truths.append(labels)                         # 新增真實標籤

    scheduler.step()                                         # 更新 lr 的參數，這裡要注意它不是放在 for loop 裡面，因為我們是算完一個 ep 才記一次 lr 。

    train_loss /= len(train_loader)                          # 計算整個 epoch 的平均損失

    predictions = torch.cat(predictions)                     # 將所有 batch 的預測結果拼成一個完整的張量。
    ground_truths = torch.cat(ground_truths)                 # 將所有的真實標籤拼成一個完整的張量。

    train_acc, train_f1, train_misclass = evaluator(predictions, ground_truths) # 計算當前 epoch 的 acc 、 F1 score 和 misclassification_rate

    return train_loss, 100*train_acc, 100*train_f1, 100*train_misclass

## Validation


In [ ]:
# TODO_7: 助教挖掉了一小部分，請你讓它可以運作起來。
def validation(model, valid_loader, criterion, device):
    model.eval()  # 設置為驗證模式
    valid_loss = 0  # 初始化 validation 的總損失
    predictions, ground_truths = [], []  # 用來存儲預測值和真實標籤

    with torch.no_grad():  # 不計算梯度
        for ID, labels, images in valid_loader:
            labels = labels.to(device)  # 將 labels 移動到指定的設備 (GPU/CPU)
            images = images.to(device)  # 將 images 移動到指定的設備 (GPU/CPU)

            # 前向傳播
            logits = model(images)  # 使用模型進行預測
            loss = criterion(logits, labels)  # 計算 loss

            valid_loss += loss.item()  # 累加損失

            # 取得預測的類別 (logits 的最大值對應的類別)
            preds = torch.argmax(logits, dim=1).cpu().numpy()

            # 將預測和真實標籤加入列表
            predictions.extend(preds)
            ground_truths.extend(labels.cpu().numpy())

    valid_loss /= len(valid_loader)  # 計算平均損失

    predictions = np.array(predictions)  # 將預測轉成 numpy 格式
    ground_truths = np.array(ground_truths)  # 將真實標籤轉成 numpy 格式

    # 計算 accuracy, F1 score 和 misclassification rate
    valid_acc, valid_f1, valid_misclass = evaluator(predictions, ground_truths)

    return valid_loss, 100*valid_acc, 100*valid_f1, 100*valid_misclass

In [ ]:
def Run():

    seed_everything(config.seed)
    device = config.device

    train_data = pd.read_csv(os.path.join(config.root, 'zip.train.csv'), header=None)

    # TODO_8: split the training dataset in a 80: 20，填寫 train_size= , test_size= 。
    train_dataset, val_dataset = train_test_split(train_data, train_size= 0.8, test_size= 0.2, random_state=config.seed)

    # TODO_9: val_dataset =
    train_dataset = HandwrittenDigits(train_dataset, mode = "train")
    val_dataset = val_dataset = HandwrittenDigits(val_dataset, mode="train")

    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)

    #criterion = nn.CrossEntropyLoss()
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)  # 增加label smoothing


    model =  SimpleNN(input_dim=256, output_dim=10)
    model.to(device)

    # TODO_10: 調整適當的 optimizer、scheduler，不調也可以，但排名不容許你這樣做。
    parameters = [p for p in model.parameters() if p.requires_grad]  # 對 model 的所有參數，抓取那些需要計算梯度的參數用於更新
    optimizer = torch.optim.AdamW(parameters, lr=config.lr, weight_decay=config.weight_decay)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size = 12, gamma = 0.1) #6,0.1

    # 參考網站: https://pytorch.org/docs/stable/optim.html

    best_val_loss = float("inf")
    history = {
        "train": {
            "loss": [],
            "acc": [],
            "f1": [],
            "misclass": []
        },
        "valid": {
            "loss": [],
            "acc": [],
            "f1": [],
            "misclass": []
        },
    }
    for epoch in range(config.epochs):
        train_loss, train_acc, train_f1, train_misclass = train_one_epoch(model, train_loader, optimizer, scheduler, criterion, device)
        valid_loss, valid_acc, valid_f1, valid_misclass = validation(model, val_loader, criterion, device)

        # Log the loss and validation result
        history["train"]["loss"].append(train_loss)
        history["train"]["acc"].append(train_acc)
        history["train"]["f1"].append(train_f1)
        history["train"]["misclass"].append(train_misclass)
        history["valid"]["loss"].append(valid_loss)
        history["valid"]["acc"].append(valid_acc)
        history["valid"]["f1"].append(valid_f1)
        history["valid"]["misclass"].append(valid_misclass)

        print(f'Epoch[{epoch+1}/{config.epochs}], Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%, Train F1: {train_f1:.2f}% | Valid Loss: {valid_loss:.4f}, Valid Acc: {valid_acc:.2f}%, Valid F1: {valid_f1:.2f}%, Valid Misclass: {valid_misclass:.2f}% | LR: {optimizer.state_dict()["param_groups"][0]["lr"]:.6f}')

        if valid_loss < best_val_loss:
            save_file = {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "epoch": epoch,
                "args": config
            }
            best_val_loss = valid_loss
            torch.save(save_file, "checkpoint.pth")

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(config.epochs), history["train"]["loss"], label='Training Loss')
    plt.plot(range(config.epochs), history["valid"]["loss"], label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Training and Validation Loss Curves')
    plt.show()

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(config.epochs), history["train"]["acc"], label='Training Acc')
    plt.plot(range(config.epochs), history["valid"]["acc"], label='Validation Acc')
    plt.xlabel('Epoch')
    plt.ylabel('Acc')
    plt.legend()
    plt.title('Training and Validation Accuracy Curves')
    plt.show()

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(config.epochs), history["train"]["f1"], label='Training F1')
    plt.plot(range(config.epochs), history["valid"]["f1"], label='Validation F1')
    plt.xlabel('Epoch')
    plt.ylabel('F1')
    plt.legend()
    plt.title('Training and Validation F1 Score Curves')
    plt.show()

    # TODO_11: 畫出 train 跟 valid 的 Misclassification Rate 圖。 (4 pts)
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(config.epochs), history["train"]["misclass"], label='Training Misclassification Rate')
    plt.plot(range(config.epochs), history["valid"]["misclass"], label='Validation Misclassification Rate')
    plt.xlabel('Epoch')
    plt.ylabel('Misclassification Rate (%)')
    plt.legend()
    plt.title('Training and Validation Misclassification Rate Curves')
    plt.show()

In [ ]:
Run()

## testing


In [ ]:
# TODO_12: 在 testing 這個 def 填空格。
def testing(model, test_loader, device):
    model.eval()
    all_ids = []
    all_preds = []

    with torch.no_grad():  # 不計算梯度
        for ID, images in test_loader:  # 從 test_loader 中讀取 ID 和圖片
            images = images.to(device)  # 將圖片移動到指定的設備 (GPU/CPU)
            logits = model(images)  # 前向傳播，得到 logits
            preds = torch.argmax(logits, dim=1)  # 取得預測的類別

            all_ids.extend(ID)  # 擴展所有 ID
            all_preds.extend(preds.cpu().numpy())  # 將預測結果轉成 numpy 並擴展

    return all_ids, all_preds

In [ ]:
seed_everything(config.seed)
device = config.device

# TODO_13: 填寫test_data、test_dataset、test_loader、model四個空格， Hint: test_loader 的 shuffle=False，不然你會出大事。
test_data = pd.read_csv(os.path.join(config.root, 'zip.test.csv'), header=None)
test_dataset = HandwrittenDigits(test_data, mode = "test")
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)

model = SimpleNN(input_dim=256, output_dim=10)
model.to(device)
best_ckpt = torch.load("checkpoint.pth", map_location=device)
model.load_state_dict(best_ckpt["model"])

ids, preds = testing(model, test_loader, device)

In [ ]:
# Get predictions
ids, preds = testing(model, test_loader, device)

# Create DataFrame
results_df = pd.DataFrame({
    'ID': ids,
    'Class': preds
})

# TODO_14: 為輸出的 csv 檔命名。
results_df.to_csv('test.digit.csv', index=False)